In [2]:
import pandas as pd
from pyexpat import features

In [3]:
data = pd.read_csv("indian_roads_dataset.csv")
data

,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,5:00,5,Sunday,1,...,low,32,high,weather,fatal,2,2,0,NaN,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,4:00,4,Sunday,1,...,high,34,low,weather,major,4,3,0,NaN,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00,13,Wednesday,0,...,low,21,medium,weather,minor,1,1,0,NaN,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00,11,Sunday,1,...,low,30,high,distraction,minor,5,2,0,NaN,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00,16,Thursday,0,...,high,24,low,distraction,minor,2,1,0,NaN,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,Bangalore,Karnataka,13.092276,77.599571,2022-09-29,3:00,3,Thursday,0,...,high,31,low,distraction,minor,3,2,0,NaN,0.10
19996,19996,Chennai,Tamil Nadu,13.172928,80.157062,2023-11-25,1:00,1,Saturday,1,...,high,20,low,distraction,major,4,2,0,NaN,0.10
19997,19997,Chennai,Tamil Nadu,12.997170,80.150724,2022-06-18,15:00,15,Saturday,1,...,high,22,low,weather,minor,3,3,0,NaN,0.10
19998,19998,Kolkata,West Bengal,22.454882,88.322213,2023-03-12,4:00,4,Sunday,1,...,high,33,low,poor road,minor,1,0,0,NaN,0.10


In [4]:
data.dtypes

accident_id            int64
city                     str
state                    str
latitude             float64
longitude            float64
date                     str
time                     str
hour                   int64
day_of_week              str
is_weekend             int64
road_type                str
lanes                  int64
traffic_signal         int64
weather                  str
visibility               str
temperature            int64
traffic_density          str
cause                    str
accident_severity        str
vehicles_involved      int64
casualties             int64
is_peak_hour           int64
festival                 str
risk_score           float64
dtype: object

In [5]:
data.drop(columns=['festival'], inplace=True)

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['weather'] = le.fit_transform(data["weather"])
data['visibility'] = le.fit_transform(data["visibility"])


In [7]:
data['accident_severity'] = le.fit_transform(data["accident_severity"])
data['cause'] = le.fit_transform(data["cause"])
data['traffic_density'] = le.fit_transform(data["traffic_density"])
data['road_type'] = le.fit_transform(data["road_type"])
data['day_of_week'] = le.fit_transform(data["day_of_week"])
data['state'] = le.fit_transform(data["state"])
data['city'] = le.fit_transform(data["city"])

In [8]:
data['date_time'] = pd.to_datetime(
    data['date'].astype(str) + ' ' + data['time'].astype(str)
)
print(data[['date', 'time', 'date_time']].head())

         date   time           date_time
0  2023-10-22   5:00 2023-10-22 05:00:00
1  2023-05-21   4:00 2023-05-21 04:00:00
2  2024-07-10  13:00 2024-07-10 13:00:00
3  2025-03-30  11:00 2025-03-30 11:00:00
4  2024-01-25  16:00 2024-01-25 16:00:00


In [9]:
data.drop(columns=['date', 'time'], inplace=True)

In [10]:
data.dtypes

accident_id                   int64
city                          int64
state                         int64
latitude                    float64
longitude                   float64
hour                          int64
day_of_week                   int64
is_weekend                    int64
road_type                     int64
lanes                         int64
traffic_signal                int64
weather                       int64
visibility                    int64
temperature                   int64
traffic_density               int64
cause                         int64
accident_severity             int64
vehicles_involved             int64
casualties                    int64
is_peak_hour                  int64
risk_score                  float64
date_time            datetime64[us]
dtype: object

In [11]:
data.shape

(20000, 22)

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [13]:
features = [
    'city',
    'state',
    'temperature',
    'traffic_signal',
    'road_type',
]
x = data[features]
y = data['risk_score']

In [14]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [15]:
from sklearn.linear_model import LinearRegression
dt=LinearRegression()
dt.fit(x_train,y_train)
y_pred = dt.predict(x_test)

In [16]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [17]:
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE:      {mean_absolute_error(y_test, y_pred):.4f}")

R² Score: -0.0009
RMSE:     0.2156
MAE:      0.1675


In [18]:
from sklearn.svm import SVR
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [19]:
from sklearn.ensemble import RandomForestRegressor
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf_reg.fit(x_train, y_train)

y_pred = rf_reg.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error: 0.0467
R2 Score: -0.0056


In [20]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

from sklearn.svm import SVR
svr_reg = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr_reg.fit(X_train_scaled, y_train)

y_pred = svr_reg.predict(X_test_scaled)

from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error: 0.0483
R2 Score: -0.0401


In [5]:
table=pd.DataFrame({'models':['Linear Regression','Randam forest','Support vector machine'],
                    'r2_score':[-0.0009,-0.0056,-0.0401],'Mean Squared error':[0.1675,0.0467,0.0483]})

In [6]:
table

,models,r2_score,Mean Squared error
0,Linear Regression,-0.0009,0.1675
1,Randam forest,-0.0056,0.0467
2,Support vector machine,-0.0401,0.0483


In [ ]:
# conclusion : Random Forest is the best model among the options as it achieves the lowest error (MSE = 0.0467) and substantially outperforms Linear Regression. However, because all $R^2$ scores are negative, all models currently predict no better than a simple mean baseline; hyperparameter tuning, feature engineering, or scaling adjustments will be necessary to achieve strong positive accuracy.